<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


# Interfacing with the IMU

A small tilt of the Duckiedrone changes both its estimated orientation and the components reported by its accelerometer. Watching those changes together provides a way to connect the measurements and rotation representations developed throughout this LX to the physical motion they describe.

[Notebook 6](./6-imu-rates-and-configuration.ipynb) established how to observe and change the delivery rate of IMU messages. The next step is to turn those messages into a display that remains readable while the Duckiedrone moves. This notebook inspects a message, completes the subscriber that extracts its values, and uses the resulting display to compare a level pose with a tilted one.

This notebook addresses the following questions:

- How can a subscriber receive frequent IMU updates while displaying one reading per second?

- How do the acceleration components and Euler angles change when the Duckiedrone tilts?

- What can the displayed values establish about its orientation and the continued delivery of measurements?

## Getting to a terminal that can see the Duckiedrone

The ROS 2 commands below run in the MAVROS2 container used in Notebook 6. If that shell is still open, reuse it; otherwise, open one as in [Reach the ROS 2 environment](./6-imu-rates-and-configuration.ipynb#reach-the-ros-2-environment).

This shell reaches the Raspberry Pi side of [Figure 1 in Notebook 5](./5-duckiedrone-imu-architecture.ipynb#figure-1). The IMU driver runs on the flight controller, while this container receives its transmitted data and exposes the ROS 2 topics, as described in [the PX4-to-MAVROS2 section](./5-duckiedrone-imu-architecture.ipynb#from-px4-to-mavros2-over-mavlink).

## Look at the data first

Before interpreting a changing display, establish what arrives while the Duckiedrone is approximately level and stationary. On a physical Duckiedrone, keep it disarmed and resting on a stable surface.

List the IMU topics:

```bash
ros2 topic list | grep imu
```

The standard interface includes `/mavros/imu/data` and `/mavros/imu/data_raw`. Use `/data` for the orientation display because it contains the flight controller's attitude estimate alongside the processed measurements.

Measure its arrival rate:

```bash
ros2 topic hz /mavros/imu/data
```

Allow several reports to appear, then stop the command with `Ctrl-C`. Compare the result with the baseline from Notebook 6; with the standard stream configuration and sufficient source updates, it should be near $50\,\mathrm{Hz}$.

Now print one message:

```bash
ros2 topic echo --once /mavros/imu/data
```

The command exits after receiving a message. If it remains waiting, stop it with `Ctrl-C` and use the checks under ["When nothing arrives"](#when-nothing-arrives) before continuing.

Read the acceleration and orientation together. For a level, stationary Duckiedrone, `linear_acceleration.z` should be near $+9.81\,\mathrm{m/s^2}$ because the accelerometer measures the upward support against gravity. The horizontal components should be comparatively small, while `angular_velocity`, expressed in $\mathrm{rad/s}$, should be near zero. Small departures reflect the pose, measurement errors, and processing discussed earlier in the LX.

The four `orientation` components encode the estimated attitude as a quaternion. Before converting them, inspect `orientation_covariance[0]`: a value of `-1` marks the orientation as unavailable. An all-zero covariance instead means that its uncertainty is unknown. As explained in Notebook 5, the covariance values supplied by this MAVROS plugin also should not be interpreted as a fresh measurement of sensor noise.

Compare this with one message from `/data_raw`:

```bash
ros2 topic echo --once /mavros/imu/data_raw
```

This topic supplies processed acceleration and angular velocity without an orientation estimate. Its unavailable-orientation marker explains why the subscriber below uses `/data`: converting the placeholder quaternion from `/data_raw` would produce angles with no physical meaning.

## Activity 1: write the subscriber

<a id="activity-1"></a>

The terminal commands reveal the message contents, but a complete message contains more information than is convenient to read while tilting the Duckiedrone. The subscriber extracts acceleration, angular velocity, and Euler angles, then presents them in a single line.

Open [packages/solution/solution/imu_subscriber.py](../packages/solution/solution/imu_subscriber.py). The two `TO IMPLEMENT` blocks identify the subscription and message-decoding tasks. In the solution version of this LX, their implementations are already present; use them to follow how each incoming message becomes a displayed reading. In the non-solution version of this LX, complete the corresponding blocks.

The surrounding code creates the node and a timer that calls `cb_log` once per second. Incoming messages follow a different path through `cb_imu`, which updates `self.latest_reading` whenever a message arrives. Keeping those paths separate allows the stored data to follow the topic while the display remains slow enough to read.

### In `__init__`, subscribe to the IMU topic

Create the subscription with `Imu` as the message type, `topic` as the topic name, `self.cb_imu` as the callback, and `qos_profile_sensor_data` as the quality-of-service profile. Store the returned subscription in `self.sub_imu`.

Pass `self.cb_imu` without parentheses so ROS 2 can invoke it when a message arrives. Calling it during initialization would not register that future behavior.

The sensor-data profile uses best-effort delivery with a queue depth of five, matching the MAVROS2 IMU publisher. Inspect the publisher's settings in the MAVROS2 container:

```bash
ros2 topic info -v /mavros/imu/data
```

Find the publisher's reliability and history settings in the output and compare them with the selected profile. This connects the subscription configuration to the interface actually supplying the measurements.

The default `topic` is `/mavros/imu/data`.

### In `cb_imu`, unpack one message

The callback receives a `sensor_msgs/Imu` message and stores its useful fields in an `ImuReading`. Copy the three `linear_acceleration` components into `accel` and the three `angular_velocity` components into `gyro`, preserving their order as $x$, $y$, and $z$.

For the orientation, pass `msg.orientation.x`, `.y`, `.z`, and `.w` to `quaternion_to_euler` in that order. The helper applies the convention developed in Notebook 4 and returns roll, pitch, and yaw in radians; the logger later converts those angles to degrees for display.

Store the message timestamp in seconds using

```python
msg.header.stamp.sec + msg.header.stamp.nanosec * 1e-9
```

and assign the completed `ImuReading` to `self.latest_reading`.

At the measured input rate, several messages can replace `self.latest_reading` between two calls to `cb_log`. The next log line therefore presents the most recently stored reading rather than every intermediate sample. This makes the display readable, but also means that it is unsuitable for reconstructing all motion between log lines.

## Activity 2: run the node on the Duckiedrone

<a id="activity-2"></a>

Save any source changes before building. In a separate base-station terminal, outside the editor and MAVROS2 container, change to the LX root and run:

```bash
dts code build -R amelia
dts code workbench -R amelia
```

Use the configured Duckiedrone name in place of `amelia`. The workbench launches the listener and displays its output; keep this terminal available to stop the run with `Ctrl-C`.

At startup, the listener reports the topic it subscribes to. Before receiving its first message, it can print `Waiting for IMU data...`; once a message arrives, each timer update prints the stored acceleration, angular velocity, and Euler angles.

The following illustrative line shows the output format. The measured values will depend on the Duckiedrone's pose and estimate:

```text
[imu_listener_node-1] [INFO] [imu_listener_node]: accel=[-0.07 -0.11 +9.75] m/s^2 | gyro=[+0.00 +0.00 -0.00] rad/s | rpy=[  -0.6   +0.4  +89.9] deg
```

In this example, the small roll and pitch angles are consistent with an approximately level pose, and the positive vertical acceleration is consistent with support against gravity. The yaw value describes the estimator's heading; it does not establish that the Duckiedrone is aligned with a known compass direction.

The timer continues displaying the last stored reading even if new messages stop arriving. Continued log output therefore needs to be interpreted alongside the topic-rate observation, especially if the displayed values stop responding to movement.

After changing the source, stop the workbench, save the changes, and repeat the build-and-workbench sequence so the running listener uses the revised code.

### When nothing arrives

If the listener remains at `Waiting for IMU data...`, first compare its reported subscription topic with the topic listing in the MAVROS2 container. A spelling or prefix mismatch can explain why the listener receives nothing even though MAVROS is publishing.

For a matching topic, check delivery independently:

```bash
ros2 topic hz /mavros/imu/data
```

If rate reports appear, the investigation moves toward the listener's configuration, runtime environment, and deployed source. Stop the measurement with `Ctrl-C` after collecting several reports.

If the command remains silent, inspect the visible ROS nodes:

```bash
ros2 node list
```

The standard setup should include `/mavros`. Its presence establishes that the node is visible, while receiving IMU messages additionally depends on its plugin, the flight-controller connection, and the upstream data. Check those stages before assigning the failure to the sensor.

For virtual operation, also confirm that the Duckiedrone is running and attached to a Duckiematrix entity. The absence of simulated source data can otherwise produce the same downstream silence.

## Activity 3: tilt the Duckiedrone and watch the angles

<a id="activity-3"></a>

Begin with the Duckiedrone level and stationary, with the listener running. On physical hardware, keep it disarmed and handle it clear of the propellers. For virtual operation, use the simulator's pose controls to produce equivalent orientations.

First, tilt about one axis and hold the new pose still. A small roll should mainly change the displayed roll angle, while a small pitch should mainly change pitch. Comparing one axis at a time makes it easier to connect the display to the body-axis convention from [Notebook 3](./3-rotation-representations.ipynb).

Now compare the acceleration components in the level and tilted poses. Once motion has settled, the support against gravity has approximately the same magnitude, but its components change because the body axes have tilted. The vertical component decreases as the body $z$ axis moves away from world vertical, while horizontal components appear.

The displayed pitch lies between $-90^\circ$ and $+90^\circ$ because the conversion selects that Euler-angle range. Near either endpoint, roll and yaw can change together or take a different equivalent representation, even while the physical orientation changes smoothly. The quaternion-to-Euler conversion discussed in Notebook 4 explains this behavior.

Finally, return the Duckiedrone to a stationary pose and watch yaw for about a minute. Record whether it changes and by how much. Without an effective heading reference, residual gyroscope error can accumulate into heading drift, although a short observation may show little change.

Stop the workbench with `Ctrl-C` after completing the observations.

## Going further

The one-second display deliberately omits intermediate readings. To examine faster changes, compare both IMU topics in a subscriber or record the messages for later analysis.

When comparing `/mavros/imu/data_raw` with `/mavros/imu/data`, inspect the acceleration, angular velocity, orientation availability, and covariance fields. Recall from Notebook 5 that the two topics have different publication triggers and processing paths, so their values need not form synchronized pairs.

To record `/data`, run the following command in the MAVROS2 container:

```bash
ros2 bag record -o /tmp/imu_tilt /mavros/imu/data
```

Perform a short tilt-and-hold sequence, then stop recording with `Ctrl-C`. The recording is written inside the container, so copy it to the base station before the container is removed. From a base-station terminal, run:

```bash
ssh duckie@amelia.local 'docker cp ros2-mavros:/tmp/imu_tilt -' | tar -xf - -C "$HOME"
```

Replace `amelia` with the configured hostname. This places the recording in `~/imu_tilt` on the base station. To replay it, run the following command in the MAVROS2 container, where playback republishes the recorded messages on their original topic:

```bash
ros2 bag play /tmp/imu_tilt
```

## Summary and conclusions

A tilt changes the estimated roll and pitch and moves part of the stationary proper acceleration from the body $z$ axis onto the horizontal axes. The fields of one IMU message differ in origin (measured or estimated), frame, and timing, and reading the display correctly depends on all three.

## Further reading

The [sensor_msgs/Imu message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Imu.msg) lists the fields unpacked by the subscriber.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()